In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y libassimp-dev
!pip install -q --upgrade "hyperdrone[examples]"

In [ ]:
import math
from pathlib import Path

import imageio.v2 as imageio
import numpy as np

from hyperdrone import dynamics, render
from hyperdrone.examples.data import procthor_scene_path

WIDTH, HEIGHT = 128, 128
STEPS, FPS = 180, 30
START = np.array([-3.92, -5.67, 1.0], dtype=np.float32)

# A tiny procedural quadrotor shared by both camera views.
CORNERS = np.array([
    [-1, -1, -1], [1, -1, -1], [-1, 1, -1], [1, 1, -1],
    [-1, -1,  1], [1, -1,  1], [-1, 1,  1], [1, 1,  1],
], dtype=np.float32)
TRIANGLES = np.array([
    [0, 1, 3], [2, 3, 0], [5, 7, 6], [5, 6, 4],
    [0, 4, 5], [0, 5, 1], [2, 3, 7], [2, 7, 6],
    [1, 5, 7], [1, 7, 3], [4, 0, 2], [4, 2, 6],
], dtype=np.int32)

def box(center, size, color):
    vertices = CORNERS * (np.asarray(size) / 2) + np.asarray(center)
    return render.Mesh(vertices.astype(np.float32), TRIANGLES, color=color)

drone = render.Object(name="drone")
for center, size, color in [
    ((0, 0, 0), (.12, .08, .05), (.15, .15, .18)),
    ((0, 0, 0), (.34, .025, .018), (.7, .7, .75)),
    ((0, 0, 0), (.025, .34, .018), (.7, .7, .75)),
    ((.15, 0, .02), (.08, .08, .012), (.95, .25, .15)),
    ((-.15, 0, .02), (.08, .08, .012), (.2, .35, .95)),
    ((0, .15, .02), (.08, .08, .012), (.2, .2, .2)),
    ((0, -.15, .02), (.08, .08, .012), (.2, .2, .2)),
]:
    drone.add_mesh(box(center, size, color))

scene = render.load_scene(procthor_scene_path(), fidelity="medium")
assets = render.AssetPool()
drone_asset = assets.add_object(drone)

sim = dynamics.Sim(num_drones=1, model="crazyflie", device="auto")
sim.reset(seed=0, sample_states=False)
sim.state["position"] = START[None]
sim.state["linear_velocity"] = np.array([[.25, 0, 0]], dtype=np.float32)

renderer = render.Renderer(
    width=WIDTH, height=HEIGHT, num_cameras=2, output="rgb", fidelity="medium",
    num_overlays=1, max_overlay_instances=1, max_overlays_per_camera=1,
)
renderer.init(scene, assets)
renderer.attach(0, 0)  # onboard camera sees its own drone
renderer.attach(1, 0)  # external camera sees the same drone
placement = renderer.spawn(0, drone_asset, render.make_transform(position=START))
renderer.update()

# Body-frame camera, just above the fuselage: arms and front rotor remain visible.
MOUNT = np.array([[1, 0, 0, .055], [0, 1, 0, 0], [0, 0, 1, .04]], dtype=np.float32)
hover = 2 * sim.parameters["hovering_throttle_relative"][0] - 1
base_action = hover + np.array([.02, -.02, .02, -.02], dtype=np.float32)


In [ ]:
frames = []
for _ in range(STEPS):
    position = sim.state.numpy("position")[0]
    velocity_z = sim.state.numpy("linear_velocity")[0, 2]
    correction = .4 * (START[2] - position[2]) - .25 * velocity_z
    sim.step(np.clip(base_action + correction, -1, 1)[None].astype(np.float32))

    position = sim.state.numpy("position")[0]
    orientation = sim.state.numpy("orientation")[0]
    transform = render.make_transform(position=position, orientation_wxyz=orientation)
    renderer.set_transform(0, placement, transform)
    renderer.update()

    onboard = sim.camera_bases_numpy(mount=MOUNT, fov=math.radians(100), aspect=renderer.aspect)
    external = renderer.camera(
        position=position + np.array([-.3, -.3, .2]),
        look_at=position,
        fov=math.radians(55),
    ).reshape(1, 12)
    renderer.set_cameras(np.concatenate([onboard, external]))
    renderer.render("rgb")
    views = renderer.frame()[..., :3]
    frames.append(np.concatenate([views[0], views[1]], axis=1))

mp4_path = Path("hyperdrone_sim.mp4")
gif_path = Path("hyperdrone_sim.gif")
imageio.mimsave(mp4_path, frames, fps=FPS)
imageio.mimsave(gif_path, frames[::2], duration=2 / FPS, loop=0)
print(f"Saved {mp4_path} and {gif_path} (onboard/self-occlusion | external chase)")

In [ ]:
from IPython.display import Image, Video, display

display(Video(str(mp4_path), embed=True))
display(Image(filename=str(gif_path)))